In [17]:
import requests
import time
import sys
import os
import json
import uuid
from psycopg import OperationalError, DatabaseError

from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
import random
from dotenv import load_dotenv
import psycopg

In [6]:
def is_elasticsearch_ready(url="http://127.0.0.1:9200"):
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Elasticsearch service: {e}")
        return False

In [7]:
is_elasticsearch_ready()

True

In [8]:
def is_grafana_ready(url="http://127.0.0.1:3000"):
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Grafana service: {e}")
        return False

In [9]:
is_grafana_ready()

Error connecting to Grafana service: HTTPConnectionPool(host='127.0.0.1', port=3000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cba109a5670>: Failed to establish a new connection: [Errno 111] Connection refused'))


False

In [10]:
def is_mage_ready(url="http://127.0.0.1:6789"):
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Mage service: {e}")
        return False

In [11]:
is_mage_ready()

True

In [12]:
def wait_for_services(max_retries=12):  # 12 * 5 seconds = 1 minute total wait time
    retries = 0
    while retries < max_retries:
        if is_elasticsearch_ready() and is_mage_ready() and is_grafana_ready():
            print("All  Elasticsearch and Mage and Grafana are ready!")
            return True
        else:
            print(f"Attempt {retries + 1}/{max_retries}: Services not ready. Waiting 5 seconds...")
            time.sleep(5)
            retries += 1
    print("Max retries reached. Services are not ready.")
    return False

In [13]:
wait_for_services()

Error connecting to Grafana service: HTTPConnectionPool(host='127.0.0.1', port=3000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cba109a5d00>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 1/12: Services not ready. Waiting 5 seconds...
Error connecting to Grafana service: HTTPConnectionPool(host='127.0.0.1', port=3000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cba109a5e50>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 2/12: Services not ready. Waiting 5 seconds...
Error connecting to Grafana service: HTTPConnectionPool(host='127.0.0.1', port=3000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cba109a5430>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 3/12: Services not ready. Wait

KeyboardInterrupt: 

In [14]:
def run_pipeline_populate_elasticsearch():
    """
    populating by our KNWLG.base
    """
    url = "http://127.0.0.1:6789/api/pipeline_schedules/1/pipeline_runs/60ac297fd34a457991914d00c79c6a42"
    
    headers = {
        "Content-Type": "application/json"
    }

    print('!----> populate_elasticsearch for magic started', flush=True)
    
    try:
        response = requests.post(url, headers=headers)
        response.raise_for_status()
        print(f'!----> populate_elasticsearch magic finished with code: {response.status_code}', flush=True)
    except Exception as err:
        print(f"An unexpected error occurred magic: {err}", flush=True)
        print("Error details magic:", sys.exc_info(), flush=True)
    finally:
        print("!----> Script execution completed magic.", flush=True)

In [15]:
run_pipeline_populate_elasticsearch()

!----> populate_elasticsearch for magic started
!----> populate_elasticsearch magic finished with code: 200
!----> Script execution completed magic.


In [18]:
def get_db_connection(localhost=False):

    if localhost:
        host=os.getenv("POSTGRES_HOST_LOCAL", "localhost")
    else:
        host=os.getenv("POSTGRES_HOST", "postgres")
    try:
        connection = psycopg.connect(
            host=host,
            database=os.getenv("POSTGRES_DB", "ecommerce_chatbot"),
            user=os.getenv("POSTGRES_USER", "user"),
            password=os.getenv("POSTGRES_PASSWORD", "password"),
        )
        return connection
    except OperationalError as e:
        print(f"Error: Could not connect to the PostgreSQL database.\nDetails: {e}")
        return None

In [20]:
get_db_connection(localhost=True)

ProgrammingError: invalid connection option "database"

Multiple connection attempts failed. All failures were:
- host: 'localhost', port: None, hostaddr: '::1': invalid connection option "database"

- host: 'localhost', port: None, hostaddr: '127.0.0.1': invalid connection option "database"
